# Visual ML Foundations: Medical Image Windowing & Intensity Scaling
### Dynamic Range Compression: Bone vs. Soft Tissue Windows

---

## 1. Concept Overview
Medical scans (CT and MRI) produce 12-bit to 16-bit grayscale values (ranging from -1000 to +3000 Hounsfield Units, or raw MRI proton density values). Computer monitors and vision models expect 8-bit values ($[0, 255]$).

If you normalize globally with `img / max`, subtle soft tissue contrasts (ligament tears, cartilage wear) disappear into black or gray mush.
We use **Window Center (Level)** and **Window Width**:
$$\text{Val}_{\text{min}} = \text{Center} - \frac{\text{Width}}{2}, \quad \text{Val}_{\text{max}} = \text{Center} + \frac{\text{Width}}{2}$$
Values outside this window are clamped, and the remaining band is mapped linearly to $[0, 1]$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def apply_windowing(image, window_center, window_width):
    """Apply radiological window level and width."""
    img_min = window_center - window_width / 2.0
    img_max = window_center + window_width / 2.0
    windowed = np.clip(image, img_min, img_max)
    return (windowed - img_min) / (img_max - img_min)

# Simulate wide dynamic-range radiologic scan (-500 to +1500 intensity)
np.random.seed(42)
y, x = np.ogrid[:200, :200]
base_scan = np.full((200, 200), -200.0) # Background fat/air

# Soft tissue muscle/capsule (-50 to +150)
soft_tissue = ((y - 100)**2 + (x - 100)**2) < 70**2
base_scan[soft_tissue] = np.random.normal(50, 25, size=np.sum(soft_tissue))

# Dense cortical bone (800 to 1400)
bone_ring = (((y - 100)**2 + (x - 100)**2) < 35**2) & (((y - 100)**2 + (x - 100)**2) > 20**2)
base_scan[bone_ring] = np.random.normal(1100, 50, size=np.sum(bone_ring))

# Compare Default Global Scaling vs. Specialized Windows
global_scaled = (base_scan - base_scan.min()) / (base_scan.max() - base_scan.min())
soft_tissue_win = apply_windowing(base_scan, window_center=50, window_width=250)
bone_win = apply_windowing(base_scan, window_center=1000, window_width=800)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(global_scaled, cmap='bone')
axes[0].set_title("Global Min-Max (Poor Contrast)", fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(soft_tissue_win, cmap='bone')
axes[1].set_title("Soft Tissue Window (C=50, W=250)", fontsize=12, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(bone_win, cmap='bone')
axes[2].set_title("Bone Window (C=1000, W=800)", fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()
print("Radiological windowing comparison rendered successfully.")
